# HINT-G
##### This notebook is for reproducing the main experimental results of the paper.

In [55]:
import torch
from utils import args

### Experiment Configuration

Set the following options to reproduce specific experimental results:

- `model_type`: Which variant of HINT-G to use — `"Node"` for **HINT-G_Node**, `"Edge"` for **HINT-G_Edge** (as in the paper).
- `dataset`: Dataset to run experiments on — choose from `"syn3"`, `"syn4"`, or `"BA-2motif"`.
- `gnn_type`: Type of GNN being explained — `"supervised"` or `"unsupervised"`.
- `task`: Type of explanation — `"pos"` for existing edges, `"neg"` for non-existing edges.
- `device`: Set to `"cuda"` for GPU or `"cpu"` for CPU execution.

**Note:** `BA-2motif` is a graph classification dataset and does not support `unsupervised` GNNs.

In [56]:
args.model_type = "Edge" # "Node" or "Edge"
args.dataset = "BA-2motif" # "syn3" or "syn4" or "BA-2motif"
args.gnn_type = 'supervised' # 'supervised' or 'unsupervised'
args.task = 'neg' # 'pos' or 'neg'

args.device = 'cuda'

if args.dataset == "BA-2motif" and args.gnn_type == "unsupervised":
    raise ValueError("Unsupervised GNN is not supported for graph classification datasets like BA-2motif.")

### LiSSA Hyperparameters

Set the parameters for approximating the inverse Hessian-vector product using LiSSA:

- `args.iter`: Number of iterations for the recursive approximation.
- `args.scale`: Inverse learning rate (used to scale the step size in recursion).

In [57]:
args.iter = 10
args.scale = 100000

In [58]:
if args.model_type =="Node":
    from models import HINT_G_Node as HINT_G
elif args.model_type =="Edge":
    from models import HINT_G_Edge as HINT_G
    
args.gnn_task = 'node' if args.dataset[:3] == 'syn' else 'graph'
args.device = torch.device('cuda' if torch.cuda.is_available() and args.device=='cuda' else 'cpu')
args.model_weight_path = args.save_path + args.dataset +'_'+ args.gnn_type

if args.dataset == "BA-2motif":
    args.hiddens="25-25-25"
    args.concat=False
    args.bn=False
if args.gnn_type =="unsupervised":
    args.hidden_dim=128

In [59]:
explainer = HINT_G(args=args)
rocauc, acc, precision, recall = explainer.edge_influences()
print(rocauc)


  0%|          | 0/200 [00:04<?, ?it/s]


KeyboardInterrupt: 